In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 3.3 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import cv2
import numpy as np
import networkx as nx
from skimage.morphology import skeletonize
from scipy.ndimage import label

def analyze_roads(image_path):
    # Load image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")

    # Threshold to get binary road network (roads are bright)
    _, roads = cv2.threshold(img, 200, 255, cv2.THRESH_BINARY)

    # Normalize to binary (0 or 1)
    roads = roads / 255.0

    # Skeletonize to reduce roads to single-pixel lines
    roads_skeleton = skeletonize(roads).astype(np.uint8)

    # Create a graph using NetworkX
    G = nx.Graph()

    # Find nodes (intersections or endpoints)
    labeled_array, num_features = label(roads_skeleton)
    nodes = []
    for i in range(1, num_features + 1):
        # Get coordinates of each labeled region
        coords = np.where(labeled_array == i)
        if len(coords[0]) > 0:
            # Use centroid or representative point as node
            node = (int(coords[0].mean()), int(coords[1].mean()))
            nodes.append(node)
            G.add_node(node)

    # Add edges by tracing connectivity in the skeleton
    for node1 in nodes:
        for node2 in nodes:
            if node1 != node2:
                # Check if there's a path between nodes in the skeleton
                # Simplified: Use Euclidean distance as a proxy for connectivity
                dist = np.sqrt((node1[0] - node2[0])**2 + (node1[1] - node2[1])**2)
                if dist < 10:  # Adjust threshold based on image scale
                    G.add_edge(node1, node2, weight=dist)

    # Calculate basic stats using NetworkX
    avg_degree = np.mean([d for n, d in G.degree()])
    edge_density = nx.density(G)

    print("Average node degree:", avg_degree)
    print("Edge density:", edge_density)


    return {"avg_degree": avg_degree, "edge_density": edge_density}

# Example usage
try:
    stats = analyze_roads("/content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/new_images2/brand_new_world_22.png")
    print(stats)
except Exception as e:
    print(f"Error: {e}")

Average node degree: 15.324356124983659
Edge density: 0.0020037076523252693
{'avg_degree': np.float64(15.324356124983659), 'edge_density': 0.0020037076523252693}
